# Projection sur spins locaux — pipeline de bout en bout

**Entree** : un fichier `ESCASOUT` d'un calcul CAS(n,m)SCF.

**Sortie** : pour *tous* les vecteurs propres du fichier, le spin total
`<S²>` / `<S_z>`, le spin local de chaque fragment `<S²_frag>`, et le poids de
chaque etat de spin local demande (singulet, triplet, doublet...).

Il suffit de remplir la cellule **Configuration**. Ms n'a pas besoin d'etre
fourni : il est detecte et seulement rapporte (l'extraction le gere
automatiquement, quel que soit Ms ou la taille du CAS).

In [1]:
from spin_projection import (
    LocalSpinAnalyzer,
    detect_orbitals, detect_n_electrons, detect_ms,
)

## Configuration

In [ ]:
# --- Fichier ESCASOUT a analyser ---------------------------------------
escasout_file = "example_datas/ESCASOUT3.ms0"
# escasout_file = "example_datas/ESCASOUT3.ms2"
# --- Fragments : orbitales (etiquettes telles que dans le fichier) + spins
#     a projeter. Spins possibles : singlet, doublet, triplet, quartet,
#     quintet, sextet, septet (ou une valeur numerique S(S+1)).
fragments = {
    "metal":  {"orbitals": ['106', '107', '108', '109'], "spins": ['singlet', 'triplet']},
    "ligand": {"orbitals": ['110', '111'],               "spins": ['singlet', 'triplet']},
}

# --- Nombre d'electrons actifs : None => detecte automatiquement --------
n_electrons = None

# --- Orbitales gelees / virtuelles (etiquettes). Vide = aucune ---------
frozen_orbitals  = []
virtual_orbitals = []

## Detection automatique (info)

In [22]:
orbitals_detectees = detect_orbitals(escasout_file)
n_elec_detecte     = detect_n_electrons(escasout_file)
ms_detecte         = detect_ms(escasout_file)

if n_electrons is None:
    n_electrons = n_elec_detecte

orbitals_config = sorted({o for frag in fragments.values() for o in frag['orbitals']}, key=int)

print(f"Fichier            : {escasout_file}")
print(f"Orbitales (fichier): {orbitals_detectees}")
print(f"Orbitales (config) : {orbitals_config}")
print(f"N electrons        : {n_electrons}")
print(f"Ms detecte         : {ms_detecte}")

if orbitals_config != orbitals_detectees:
    print("\n[ATTENTION] Les orbitales des fragments ne correspondent pas a celles")
    print("            trouvees dans le fichier. Verifie la configuration.")

Fichier            : ../NB139/-.e7/ESCASOUT3.ms0
Orbitales (fichier): ['106', '107', '108', '109', '110', '111']
Orbitales (config) : ['106', '107', '108', '109', '110', '111']
N electrons        : 6
Ms detecte         : 0.0


## Construction de l'analyseur et analyse de tous les vecteurs

In [23]:
analyzer = LocalSpinAnalyzer(
    n_electrons=n_electrons,
    fragments=fragments,
    frozen_orbitals=frozen_orbitals,
    virtual_orbitals=virtual_orbitals,
)

results = analyzer.analyze_file(escasout_file)
results

,S2_total,Sz,S2_metal,metal_singlet,metal_triplet,S2_ligand,ligand_singlet,ligand_triplet
v1,2.0,0.0,0.030,0.961,0.001,1.952,0.000,0.962
v2,12.0,0.0,6.000,0.000,0.000,2.000,0.000,1.000
v3,2.0,0.0,5.745,0.000,0.005,1.862,0.004,0.891
v4,2.0,0.0,1.973,0.001,0.963,0.059,0.949,0.016
v5,2.0,0.0,1.946,0.001,0.947,1.896,0.020,0.928


## (Optionnel) Inspecter un vecteur en particulier

In [6]:
# Affiche les determinants dominants d'un vecteur donne.
_ = analyzer.visualize_vector(escasout_file, vec_num=3, cutoff=0.1)


	-----------
	 Coeff.     N-body state and index 
	-------     ----------------------
	+0.25964  ||βββααα⟩⟩    #628 
	+0.25964  ||αααβββ⟩⟩    #295 
	+0.25867  ||βαββαα⟩⟩    #564 
	+0.25867  ||αβααββ⟩⟩    #359 
	+0.25850  ||αβββαα⟩⟩    #382 
	+0.25850  ||βαααββ⟩⟩    #541 
	+0.25684  ||ββαβαα⟩⟩    #614 
	+0.25684  ||ααβαββ⟩⟩    #309 



## (Optionnel) Sauvegarder le tableau de resultats

In [6]:
# results.to_csv("resultats_spin_local.csv")